# Carnatic Music Processing

## Outline / Steps

* Import data
  * Carnatic Varnam dataset audio recordings
* Preprocessing - visualize audio
  * Function 1: Get spectrogram
  * Function 2: Get fundamental pitch contour
  * Function 3: Separation of drone / vocal?
  * Function 4: Annotate percussive onsets
* Feature Calculations
  * Function 5: Plot pitch contour derivative + mark saddle points
  * Function 6: Overall Pitch Statistics (histogram + probability density function)
* Sandbox

* Framework
- parameter + results ()
  - median in addition to mean
- Github
  - clear all outputs
  - setup to commit


## Test File

### Setup

In [ ]:
# IMPORTS

import os
import librosa as lb
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import medfilt
from scipy.interpolate import interp1d
import scipy.stats as stats


In [ ]:
# LOAD FILE

abhogi1 = 'carnatic_varnam_1.0/Audio/223578__gopalkoduri__carnatic-varnam-by-dharini-in-abhogi-raaga.mp3'
x, sr = lb.load(abhogi1)

### Spectrogram + Pitch Contour Generation

ISSUE: f0 and spectrogram overlay takes forever if mismatch in timesteps, gave them both the same hop_length parameter but play around with this

In [ ]:
spec = lb.amplitude_to_db(np.abs(lb.stft(x, n_fft=2048, hop_length=512)), ref=np.max)
f0, voiced_flag, voiced_probs = lb.pyin(x, fmin=lb.note_to_hz('C2'), fmax=lb.note_to_hz('C7'), hop_length=512)

In [ ]:
# Plot spectrogram
plt.figure(figsize=(12, 6))

plt.subplot(2, 1, 1)
lb.display.specshow(spec, sr=sr, x_axis='time', y_axis='log')
plt.colorbar(format='%+2.0f dB')
plt.title('Spectrogram')

# Plot f0 contour
plt.subplot(2, 1, 2)
times = lb.times_like(f0, sr=sr)
plt.plot(times, f0, label='f0 contour', color='r')
plt.xlabel('Time (s)')
plt.ylabel('Frequency (Hz)')
plt.title('Fundamental Frequency (f0) Contour')
plt.legend()

plt.tight_layout()
plt.show()

ISSUE: slow in general, will start with just ~20 seconds (1000 samples w curr params) of each sample as the default


In [ ]:
spec_clip = spec[:, :1000]
f0_clip = f0[:1000]

In [ ]:
# Plot spectrogram
plt.figure(figsize=(12, 6))

plt.subplot(2, 1, 1)
lb.display.specshow(spec_clip, sr=sr, x_axis='time', y_axis='log')
plt.colorbar(format='%+2.0f dB')
plt.title('Spectrogram')

# Plot f0 contour
plt.subplot(2, 1, 2)
times = lb.times_like(f0_clip, sr=sr)
plt.plot(times, f0_clip, label='f0 contour', color='r')
plt.xlabel('Time (s)')
plt.ylabel('Frequency (Hz)')
plt.title('Fundamental Frequency (f0) Contour')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
times = lb.times_like(f0_clip, sr=sr, hop_length=512)

# Plot spectrogram
plt.figure(figsize=(12, 6))
lb.display.specshow(spec_clip, sr=sr, hop_length=512, x_axis='time', y_axis='log')
plt.colorbar(format='%+2.0f dB')

# Overlay f0 contour
plt.plot(times, f0_clip, label='f0 contour', color='r', linewidth=2)

# Labels and title
plt.xlabel('Time (s)')
plt.ylabel('Frequency (Hz)')
plt.title('Spectrogram with f0 Contour')
plt.legend()

plt.show()

ISSUE: lots of noise (see overlay) -- how to remove?

In [ ]:
# Apply median filtering to remove noise
f0_filtered = medfilt(f0_clip, kernel_size=3)

# Remove out-of-bounds values
fmin_hz = 110 # ranges from cepstrum paper
fmax_hz = 250
f0_filtered = np.where((f0_filtered > fmin_hz) & (f0_filtered < fmax_hz), f0_filtered, np.nan)

# Interpolate to fill missing values
times = lb.times_like(f0_clip, sr=sr, hop_length=512)
valid = ~np.isnan(f0_filtered)
interp_func = interp1d(times[valid], f0_filtered[valid], kind='linear', fill_value="extrapolate")
f0_smooth = interp_func(times)

# Plot spectrogram
plt.figure(figsize=(12, 6))
lb.display.specshow(spec_clip, sr=sr, hop_length=512, x_axis='time', y_axis='log')
plt.colorbar(format='%+2.0f dB')

# Overlay f0 contour
plt.plot(times, f0_smooth, label='Cleaned f0 Contour', color='r', linewidth=2)

# Labels and title
plt.xlabel('Time (s)')
plt.ylabel('Frequency (Hz)')
plt.title('Spectrogram with Cleaned f0 Contour')
plt.legend()

plt.show()

### Pitch Contour Features

1. Pitch Contour Derivative
2. "Saddle" Points / Local Optima (Derivative = 0)
* derivative contour has many spikes / jumps  -- tried smoothing with a convolution (similar to median filtering and smoothing of f0)
* zero crossings are still not marked at the exact locations because they are often between samples -- tried linear interpolation for markers and timestamps
3. Onsets (for Note Boundaries)


In [ ]:
def smooth_signal(signal, kernel_size=5):
    """Apply a moving average filter to smooth the signal."""
    kernel = np.ones(kernel_size) / kernel_size
    return np.convolve(signal, kernel, mode='same')

# Compute derivative of f0
f0_derivative = np.gradient(f0_smooth, times)# Smooth the derivative to remove noise
f0_derivative_smooth = smooth_signal(f0_derivative, kernel_size=7)


# Find sign changes
sign_changes = np.diff(np.sign(f0_derivative_smooth))

# Find indices where sign change is significant
zero_crossings = np.where(np.abs(sign_changes) == 2)[0]  # Only strong sign flips

# OPTIONAL: Further filter to ignore rapid oscillations (set minimum spacing)
min_spacing = 5  # Adjust as needed based on hop_length and sampling rate
zero_crossings = zero_crossings[np.diff(np.concatenate(([0], zero_crossings))) > min_spacing]

# Plot spectrogram
plt.figure(figsize=(12, 8))

# Plot spectrogram with f0 contour
plt.subplot(2, 1, 1)
lb.display.specshow(spec_clip, sr=sr, hop_length=512, x_axis='time', y_axis='log')
plt.colorbar(format='%+2.0f dB')
plt.plot(times, f0_smooth, label='Cleaned f0 Contour', color='r', linewidth=2)
plt.scatter(times[zero_crossings], f0_smooth[zero_crossings], color='b', marker='o', label='Zero Derivative Points')
plt.xlabel('Time (s)')
plt.ylabel('Frequency (Hz)')
plt.title('Spectrogram with Cleaned f0 Contour')
plt.legend()

# Plot derivative of f0
plt.subplot(2, 1, 2)
plt.plot(times, f0_derivative, color='gray', alpha=0.5, label='Raw f0 Derivative')
plt.plot(times, f0_derivative_smooth, color='g', label='Smoothed f0 Derivative')
plt.axhline(0, color='k', linestyle='--', linewidth=1)  # Reference line at 0
plt.scatter(times[zero_crossings], f0_derivative_smooth[zero_crossings], color='b', marker='o', label='Zero Derivative Points')
plt.xlabel('Time (s)')
plt.ylabel('Frequency Change (Hz/s)')
plt.title('Derivative of f0 Contour')
plt.legend()

plt.tight_layout()
plt.show()


4. Pitch Histogram
* in terms of midi and not frequency to be scaled for perceived pitch (can also display on a log scale)
5. Pitch PDF
* uses kernel density estimation (built-in) with a Gaussian kernel

In [ ]:
f0_clean = np.interp(times, times[valid], f0_filtered[valid])

# Convert f0 to MIDI note numbers
pitch_midi = lb.hz_to_midi(f0_clean)

# Plot pitch histogram
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.hist(pitch_midi, bins=60, density=True, alpha=0.7, color='b', edgecolor='black')
plt.xlabel('MIDI Note')
plt.ylabel('Probability Density')
plt.title('Pitch Histogram')

# Compute Kernel Density Estimation (KDE)
kde = stats.gaussian_kde(pitch_midi)
pitch_range = np.linspace(min(pitch_midi), max(pitch_midi), 500)
kde_values = kde(pitch_range)

# Plot KDE-based PDF
plt.subplot(1, 2, 2)
plt.plot(pitch_range, kde_values, color='r', linewidth=2)
plt.xlabel('MIDI Note')
plt.ylabel('Density')
plt.title('Pitch Density (KDE)')

plt.tight_layout()
plt.show()


## Cleaned Functions

In [ ]:
def load_dataset(directory_path):
    data = []
    try:
        items = os.listdir(directory_path)
    except FileNotFoundError:
        print(f"Error: Directory '{directory_path}' not found.")
        exit()
    for item in items:
        data.append(lb.load(item))
    return data

In [ ]:
# def plot_spec(spec, sr, hop_length = 512, f0 = None, times = None, subplot = False):
    
#     # Plot spectrogram
#     if not subplot:
#         plt.figure(figsize=(12, 6))
#     lb.display.specshow(spec, sr=sr, hop_length=hop_length, x_axis='time', y_axis='log')
#     plt.colorbar(format='%+2.0f dB')

#     if f0 != None:
#         # Overlay f0 contour
#         plt.plot(times, f0, label='Cleaned f0 Contour', color='r', linewidth=2)

#     # Labels and title
#     plt.xlabel('Time (s)')
#     plt.ylabel('Frequency (Hz)')
#     plt.title('Spectrogram')
#     plt.legend()

#     plt.show()

#     return None

In [ ]:

def plot_spec(spec, sr, hop_length=512, f0=None, times=None, subplot=False):
    if not subplot:
        plt.figure(figsize=(12, 6))

    # Plot spectrogram
    lb.display.specshow(spec, sr=sr, hop_length=hop_length, x_axis='time', y_axis='log')

    if f0 is not None and times is not None:
        print("Spec shape:", spec.shape)  # Should be (1025, 250)
        print("f0 shape:", f0.shape)  # Should match 250
        print("times shape:", times.shape)  # Should match 250
        print("Number of NaN values in f0:", np.isnan(f0).sum())

        # Ensure `times` is generated correctly to match `spec`
        times = np.arange(spec.shape[1]) * hop_length / sr  # Ensure times match the frames

        # Handle NaNs in f0, if any
        if np.isnan(f0).sum() > 0:
            valid = ~np.isnan(f0)
            interp_func = interp1d(times[valid], f0[valid], kind='linear', fill_value="extrapolate")
            f0 = interp_func(times)

        plt.plot(times, f0, label='Cleaned f0 Contour', color='r', linewidth=2)

    plt.colorbar(format='%+2.0f dB')
    plt.xlabel('Time (s)')
    plt.ylabel('Frequency (Hz)')
    plt.title('Spectrogram')
    plt.legend()
    plt.show()

In [ ]:
def f0_filter(f0, kernel_size = 3):
    # Apply median filtering to remove noise
    f0_filtered = medfilt(f0, kernel_size=kernel_size)

    # Remove out-of-bounds values
    fmin_hz = 110 # ranges from cepstrum paper
    fmax_hz = 250
    f0_filtered = np.where((f0_filtered > fmin_hz) & (f0_filtered < fmax_hz), f0_filtered, np.nan)
    return f0_filtered

def gen_contour(x, sr, n_fft = 2048, hop_length = 512, medfilt = True, kernel_size = 3, display = True, clip_len = -1):
    
    spec = lb.amplitude_to_db(np.abs(lb.stft(x, n_fft=n_fft, hop_length=hop_length)), ref=np.max)
    f0, voiced_flags, voiced_probs = lb.pyin(x, fmin=lb.note_to_hz('C2'), fmax=lb.note_to_hz('C7'), hop_length=hop_length)

    if clip_len != -1:
        spec_clip = spec[:, :clip_len]
        f0_clip = f0[:clip_len]
    else:
        spec_clip = spec
        f0_clip = f0

    if medfilt:
        f0_filtered = f0_filter(f0_clip, kernel_size)
    else:
        f0_filtered = f0

    # Interpolate to fill missing values
    times = lb.times_like(f0_clip, sr=sr, hop_length=512)
    valid = ~np.isnan(f0_filtered)
    interp_func = interp1d(times[valid], f0_filtered[valid], kind='linear', fill_value="extrapolate")
    f0_smooth = interp_func(times)

    if display:
        print("Spectrogram shape:", spec_clip.shape)  # (freq_bins, time_steps)
        print("Times shape:", times.shape)  # Should match time_steps
        print("f0 shape:", f0_smooth.shape)  # Should match time_steps

        print("Number of NaN values in f0:", np.isnan(f0_smooth).sum())

        plot_spec(spec_clip, f0_smooth, times)

    return spec_clip, f0_smooth

    
def gen_contour_set(data, n_fft = 2048, hop_length = 512, medfilt = True, kernel_size = 3, display = True, clip_len = -1):
    specs = []
    f0s = []
    for x, sr in data:
        spec, f0 = gen_contour(x, sr, n_fft, hop_length, medfilt, kernel_size, display, clip_len)
        specs.append(spec)
        f0s.append(f0)
    return specs, f0s

In [ ]:
def smooth_signal(signal, kernel_size=5):
    """Apply a moving average filter to smooth the signal."""
    kernel = np.ones(kernel_size) / kernel_size
    return np.convolve(signal, kernel, mode='same')

def get_derivative(f0, times, smooth = True, kernel_size = 7):
    # Compute derivative of f0
    f0_derivative = np.gradient(f0, times) # Smooth the derivative to remove noise
    if smooth:
        f0_derivative_smooth = smooth_signal(f0_derivative, kernel_size=kernel_size)
    else:
        f0_derivative_smooth = f0_derivative
    return f0_derivative, f0_derivative_smooth

def get_zero_crossings(f0_der, min_spacing = 5):
    # Find sign changes
    sign_changes = np.diff(np.sign(f0_der))
    # Find indices where sign change is significant
    zero_crossings = np.where(np.abs(sign_changes) == 2)[0]  # Only strong sign flips
    # OPTIONAL: Further filter to ignore rapid oscillations (set minimum spacing)
    zero_crossings = zero_crossings[np.diff(np.concatenate(([0], zero_crossings))) > min_spacing]
    return zero_crossings

def plot_der(f0, sr, times, spec, hop_length = 512):
    f0_derivative, f0_derivative_smooth = get_derivative(f0, times)
    zero_crossings = get_zero_crossings(f0_derivative_smooth)
    # Plot spectrogram
    plt.figure(figsize=(12, 8))


    # Plot spectrogram with f0 contour
    plt.subplot(2, 1, 1)
    plot_spec(spec, sr, hop_length = 512, f0 = None, times = None, subplot = True)
    plt.scatter(times[zero_crossings], f0[zero_crossings], color='b', marker='o', label='Zero Derivative Points')

    # Plot derivative of f0
    plt.subplot(2, 1, 2)
    
    plt.plot(times, f0_derivative, color='gray', alpha=0.5, label='Raw f0 Derivative')
    plt.plot(times, f0_derivative_smooth, color='g', label='Smoothed f0 Derivative')
    plt.axhline(0, color='k', linestyle='--', linewidth=1)  # Reference line at 0
    plt.scatter(times[zero_crossings], f0_derivative_smooth[zero_crossings], color='b', marker='o', label='Zero Derivative Points')
    plt.xlabel('Time (s)')
    plt.ylabel('Frequency Change (Hz/s)')
    plt.title('Derivative of f0 Contour')
    plt.legend()


    plt.tight_layout()
    plt.show()
    return f0_derivative, f0_derivative_smooth, zero_crossings

In [ ]:
def get_pitch_dist(f0, times, filt_ker = 3, display = True):
    f0_filtered = f0_filter(f0, filt_ker)
    valid = ~np.isnan(f0_filtered)
    f0_clean = np.interp(times, times[valid], f0_filtered[valid])

    # Convert f0 to MIDI note numbers
    pitch_midi = lb.hz_to_midi(f0_clean)

    # Compute Kernel Density Estimation (KDE)
    kde = stats.gaussian_kde(pitch_midi)
    pitch_range = np.linspace(min(pitch_midi), max(pitch_midi), 500)
    kde_values = kde(pitch_range)

    if display:
        # Plot pitch histogram
        plt.figure(figsize=(12, 6))

        plt.subplot(1, 2, 1)
        plt.hist(pitch_midi, bins=60, density=True, alpha=0.7, color='b', edgecolor='black')
        plt.xlabel('MIDI Note')
        plt.ylabel('Probability Density')
        plt.title('Pitch Histogram')

        # Plot KDE-based PDF
        plt.subplot(1, 2, 2)
        plt.plot(pitch_range, kde_values, color='r', linewidth=2)
        plt.xlabel('MIDI Note')
        plt.ylabel('Density')
        plt.title('Pitch Density (KDE)')

        plt.tight_layout()
        plt.show()
    return pitch_midi, pitch_range, kde_values

## Note Transcription (2020)

Steps:
1. obtain audio recordings for ragas in the table (waiting on dataset from authors perhaps)
2. extract f0 using pyin
3. average pitch value for every frame (25 ms each, 50% overlap)
4. obtain PDF from values using Gaussian KDE
5. account for min/max values in bandwidth range (?)
6. smooth PDF with 5-point running average
7. identify tonic/base note (least variance & highest mean in 100-250 Hz range)
8. Plot derivative of pitch contour from phase I
9. Calculate onset derivative threshold (average derivative magnitude)
10. Segment by onsets
11. Label every segment

### Additional Functions

In [ ]:
def compute_average_pitch_per_frame(y, f0, sr):

    # Define frame length and hop length (50% overlap)
    frame_length = int(0.025 * sr)  # 25ms window
    hop_length = frame_length // 2  # 50% overlap

    # Compute times for each frame
    times = lb.times_like(f0, sr=sr, hop_length=512)

    # Compute average f0 per frame
    avg_f0_per_frame = []
    num_frames = (len(f0) - frame_length) // hop_length + 1

    for i in range(num_frames):
        start = i * hop_length
        end = start + frame_length

        # Extract current frame's f0 values (ignore NaNs)
        frame_f0 = f0[start:end]
        frame_f0 = frame_f0[~np.isnan(frame_f0)]  # Remove NaNs

        # Compute mean f0 if valid values exist, otherwise set NaN
        avg_f0_per_frame.append(np.mean(frame_f0) if len(frame_f0) > 0 else np.nan)

    # Compute times for each averaged frame
    avg_f0_times = np.linspace(0, len(y) / sr, len(avg_f0_per_frame))

    return avg_f0_times, avg_f0_per_frame


In [ ]:
def segment_pitch_contour(f0, f0_derivative, threshold=5):
    """Segments the pitch contour where the absolute derivative exceeds the threshold.
    
    Args:
        f0 (array): Fundamental frequency contour (Hz).
        f0_derivative (array): Derivative of the pitch contour (Hz/s).
        threshold (float): Threshold for segmentation (Hz/s).

    Returns:
        list of arrays: Segmented pitch contour pieces.
    """
    # Find indices where the derivative exceeds the threshold
    breakpoints = np.where(np.abs(f0_derivative) > threshold)[0]

    # Split f0 into segments
    segments = []
    start_idx = 0

    for bp in breakpoints:
        if bp > start_idx:  # Avoid empty segments
            segments.append(f0[start_idx:bp])
        start_idx = bp + 1  # Start next segment after the breakpoint

    # Append the last segment if it's not empty
    if start_idx < len(f0):
        segments.append(f0[start_idx:])

    return segments

### Sample Data Test

In [ ]:
kalyani1 = 'carnatic_varnam_1.0/Audio/223586__gopalkoduri__carnatic-varnam-by-badrinarayanan-in-kalyani-raaga.mp3'
k, ksr = lb.load(kalyani1)


In [ ]:
print(len(k), ksr)

In [ ]:
spec, f0 = gen_contour(k, ksr, display = True, clip_len = 250)

In [ ]:
# Plot spectrogram
plt.figure(figsize=(12, 6))

plt.subplot(2, 1, 1)
lb.display.specshow(spec, sr=sr, x_axis='time', y_axis='log')
plt.colorbar(format='%+2.0f dB')
plt.title('Spectrogram')

# Plot f0 contour
plt.subplot(2, 1, 2)
times = lb.times_like(f0_clip, sr=sr)
plt.plot(times, f0, label='f0 contour', color='r')
plt.xlabel('Time (s)')
plt.ylabel('Frequency (Hz)')
plt.title('Fundamental Frequency (f0) Contour')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
f_avg_times, f_avg_vals = compute_average_pitch_per_frame(k, f0, ksr)

In [ ]:
midis, range, kde = get_pitch_dist(f_avg_vals, f_avg_times, 3)

In [ ]:
kde_smooth = smooth_signal(kde)

In [ ]:
times = lb.times_like(f0, sr=ksr, hop_length=512)
f0_der, f0_der_smooth, zcs = plot_der(f0, ksr, times, spec)

In [ ]:
threshold = np.mean(np.abs(f0_der_smooth)) # should we use original der here?

In [ ]:
segments = segment_pitch_contour(f0, f0_der_smooth, threshold) # are the derivative and og f0 the same length?